In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Ekstraksi data dari file .mseed ke format JSON siap untuk MCU-Quake.
- Menggunakan STA/LTA untuk deteksi P-wave arrival.
- Ekstrak 7 detik sinyal dan noise.
- Preprocessing: detrend, resample ke 100 Hz, normalisasi.
- Output: JSON dengan key 'Z' dan 'Z_noise'.
"""

import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from obspy import read, UTCDateTime
from obspy.signal.trigger import recursive_sta_lta
from tqdm import tqdm
import logging

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon"
OUTPUT_JSON = "/Volumes/Extreme SSD/unduhan_waveform_geofon/extracted_data.json"

# Parameter preprocessing
SAMPLE_RATE = 100.0
SIG_DURATION = 7.0
NOISE_DURATION = 7.0
NORM_WINDOW = 9.0

# Parameter STA/LTA
STA_WIN = 1.0
LTA_WIN = 10.0
TRIGGER_THRESHOLD = 3.5

# Jumlah file untuk testing (None untuk semua)
MAX_FILES = None  # misal 100 untuk testing

# =============================================
# 2. SETUP LOGGING
# =============================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("extract_waveforms.log")
    ]
)
logger = logging.getLogger(__name__)

# =============================================
# 3. FUNGSI PREPROCESSING
# =============================================

def pick_p_arrival(trace, search_window=15):
    """
    Deteksi P-wave arrival menggunakan STA/LTA.
    Kembalikan UTCDateTime dari pick pertama.
    """
    try:
        # STA/LTA hanya pada trace yang dipotong dari awal
        tr = trace.copy()
        sr = tr.stats.sampling_rate
        sta_n = int(STA_WIN * sr)
        lta_n = int(LTA_WIN * sr)
        
        # Jika trace terlalu pendek, skip
        if len(tr.data) < lta_n + sta_n:
            return tr.stats.starttime + 5.0  # fallback
        
        cft = recursive_sta_lta(tr.data, sta_n, lta_n)
        trigger_indices = np.where(cft > TRIGGER_THRESHOLD)[0]
        
        if len(trigger_indices) > 0:
            pick_idx = trigger_indices[0]
            if pick_idx > int(2 * sr):  # hindari trigger di awal
                return tr.stats.starttime + pick_idx / sr
        
        # Fallback: ambil puncak maksimum
        max_idx = np.argmax(np.abs(tr.data))
        if max_idx > 0:
            return tr.stats.starttime + max_idx / sr
    except Exception as e:
        logger.debug(f"Picking error: {e}")
        return trace.stats.starttime + 5.0
    
    return trace.stats.starttime + 5.0

def extract_mcuquake_windows(trace, p_arrival_time):
    """
    Ekstraksi 7 detik sinyal dan noise, preprocessing.
    Return (signal_list, noise_list) atau (None, None) jika gagal.
    """
    try:
        # Potong sinyal 7 detik setelah P
        sig_start = p_arrival_time
        sig_end = p_arrival_time + SIG_DURATION
        tr_signal = trace.copy().trim(sig_start, sig_end)
        
        # Potong noise 7 detik sebelum P
        noise_start = p_arrival_time - NOISE_DURATION
        noise_end = p_arrival_time
        tr_noise = trace.copy().trim(noise_start, noise_end)
        
        # Detrend
        tr_signal.detrend('simple')
        tr_noise.detrend('simple')
        
        # Resample ke 100 Hz
        if tr_signal.stats.sampling_rate != SAMPLE_RATE:
            tr_signal.resample(SAMPLE_RATE)
        if tr_noise.stats.sampling_rate != SAMPLE_RATE:
            tr_noise.resample(SAMPLE_RATE)
        
        # Normalisasi dengan max absolut 9 detik setelah P
        tr_norm = trace.copy().trim(p_arrival_time, p_arrival_time + NORM_WINDOW)
        if len(tr_norm.data) > 0:
            max_val = np.max(np.abs(tr_norm.data))
        else:
            max_val = np.max(np.abs(tr_signal.data))
        if max_val == 0:
            max_val = 1.0
        
        signal_data = tr_signal.data / max_val
        noise_data = tr_noise.data / max_val
        
        # Potong/padding ke 700 sampel
        target_len = int(SAMPLE_RATE * SIG_DURATION)
        def fix_length(data):
            if len(data) > target_len:
                return data[:target_len]
            elif len(data) < target_len:
                return np.pad(data, (0, target_len - len(data)), 'constant')
            return data
        
        return fix_length(signal_data).tolist(), fix_length(noise_data).tolist()
    except Exception as e:
        logger.debug(f"Extraction error: {e}")
        return None, None

def process_file(file_path):
    """
    Proses satu file .mseed, return dict hasil atau None.
    """
    try:
        st = read(str(file_path))
        if len(st) == 0:
            return None
        
        # Ambil komponen Z (prioritas BHZ, HHZ, EHZ)
        trace_z = None
        for tr in st:
            if tr.stats.channel.endswith('Z'):
                trace_z = tr
                break
        if trace_z is None:
            # Jika tidak ada Z, ambil trace pertama
            trace_z = st[0]
            logger.debug(f"{file_path.name}: Tidak ada komponen Z, pakai {trace_z.stats.channel}")
        
        # Deteksi P-wave
        p_time = pick_p_arrival(trace_z)
        
        # Ekstraksi
        signal, noise = extract_mcuquake_windows(trace_z, p_time)
        if signal is None or noise is None:
            return None
        
        # Ambil info dari nama file: GE_TNTI_20100101_044258.mseed
        parts = file_path.stem.split('_')
        if len(parts) >= 3:
            network = parts[0]
            station = parts[1]
            event_id = '_'.join(parts[2:])
        else:
            event_id = file_path.stem
        
        return {
            'event_id': event_id,
            'network': network if 'network' in locals() else 'UNK',
            'station': station if 'station' in locals() else 'UNK',
            'Z': signal,
            'Z_noise': noise,
            'p_arrival': str(p_time),
            'file': file_path.name
        }
    except Exception as e:
        logger.debug(f"Error processing {file_path.name}: {e}")
        return None

def main():
    logger.info("="*60)
    logger.info("🚀 EKSTRAKSI WAVEFORM KE JSON (MCU-QUAKE)")
    logger.info("="*60)
    
    # Cari semua file .mseed
    wave_dir = Path(WAVEFORM_DIR)
    all_files = list(wave_dir.glob("*.mseed"))
    logger.info(f"📁 Ditemukan {len(all_files)} file .mseed")
    
    if MAX_FILES and len(all_files) > MAX_FILES:
        all_files = all_files[:MAX_FILES]
        logger.info(f"⚠️ Hanya memproses {MAX_FILES} file pertama.")
    
    # Load JSON yang sudah ada (resume)
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r') as f:
            existing_data = json.load(f)
        logger.info(f"📂 Load JSON existing: {len(existing_data)} entries")
    else:
        existing_data = {}
    
    # Proses file
    success = 0
    failed = 0
    skipped = 0
    
    for file_path in tqdm(all_files, desc="Memproses"):
        # Cek apakah file sudah ada di JSON
        # Gunakan nama file sebagai key (atau bagian dari event_id)
        if file_path.stem in existing_data:
            skipped += 1
            continue
        
        result = process_file(file_path)
        if result:
            # Gunakan event_id + stasiun sebagai key agar unik
            key = f"{result['event_id']}_{result['station']}"
            existing_data[key] = {
                'type': 'se',
                'Z': result['Z'],
                'Z_noise': result['Z_noise'],
                'metadata': {
                    'network': result['network'],
                    'station': result['station'],
                    'p_arrival': result['p_arrival'],
                    'file': result['file']
                }
            }
            success += 1
        else:
            failed += 1
        
        # Simpan setiap 100 file untuk menghindari kehilangan data
        if (success + failed) % 100 == 0:
            with open(OUTPUT_JSON, 'w') as f:
                json.dump(existing_data, f, indent=2)
    
    # Simpan final
    with open(OUTPUT_JSON, 'w') as f:
        json.dump(existing_data, f, indent=2)
    
    logger.info("="*60)
    logger.info(f"✨ SELESAI! Berhasil: {success}, Gagal: {failed}, Skipped: {skipped}")
    logger.info(f"📁 Total data di JSON: {len(existing_data)}")
    logger.info(f"📂 Output: {OUTPUT_JSON}")
    logger.info("="*60)

if __name__ == "__main__":
    main()

2026-06-21 05:32:55,012 - INFO - ============================================================
2026-06-21 05:32:55,012 - INFO - 🚀 EKSTRAKSI WAVEFORM KE JSON (MCU-QUAKE)
2026-06-21 05:32:55,012 - INFO - ============================================================
2026-06-21 05:32:55,073 - INFO - 📁 Ditemukan 25880 file .mseed
2026-06-21 05:32:56,359 - INFO - 📂 Load JSON existing: 3743 entries


Memproses:  42%|████▏     | 10988/25880 [07:12<06:03, 40.96it/s]/opt/homebrew/Caskroom/miniforge/base/envs/tf-metal/lib/python3.11/site-packages/obspy/signal/detrend.py:31: RuntimeWarning: invalid value encountered in divide
  data -= x1 + np.arange(ndat) * (x2 - x1) / float(ndat - 1)
Memproses: 100%|██████████| 25880/25880 [28:04<00:00, 15.36it/s]


2026-06-21 06:01:12,173 - INFO - ============================================================
2026-06-21 06:01:12,174 - INFO - ✨ SELESAI! Berhasil: 12939, Gagal: 12941, Skipped: 0
2026-06-21 06:01:12,174 - INFO - 📁 Total data di JSON: 12939
2026-06-21 06:01:12,175 - INFO - 📂 Output: /Volumes/Extreme SSD/unduhan_waveform_geofon/extracted_data.json
2026-06-21 06:01:12,175 - INFO - ============================================================


In [1]:
#!/usr/bin/env python3 GELOMBANG KEDUA
# -*- coding: utf-8 -*-
"""
EKSTRAKSI WAVEFORM 20.232 FILE KE JSON (MCU-QUAKE)
- Parallel processing (4 worker)
- Resume otomatis
- STA/LTA P-wave picker
- Preprocessing sesuai protokol Zhi Geng
"""

import os
import sys
import json
import numpy as np
from pathlib import Path
from obspy import read, UTCDateTime
from obspy.signal.trigger import recursive_sta_lta
from tqdm import tqdm
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import pandas as pd

# =============================================
# 1. KONFIGURASI
# =============================================

# Daftar file dari hasil merge
FILE_LIST_CSV = "/Volumes/Extreme SSD/unduhan_waveform_merged/file_list_merged_final.csv"
OUTPUT_JSON = "/Volumes/Extreme SSD/unduhan_waveform_merged/extracted_data.json"
LOG_FILE = "extract_waveforms.log"

# Parameter MCU-Quake
SAMPLE_RATE = 100.0
SIG_DURATION = 7.0
NOISE_DURATION = 7.0
NORM_WINDOW = 9.0

# Parameter STA/LTA
STA_WIN = 1.0
LTA_WIN = 10.0
TRIGGER_THRESHOLD = 3.5

# Parallel
MAX_WORKERS = 4

# =============================================
# 2. SETUP LOGGING
# =============================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG_FILE), logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# =============================================
# 3. FUNGSI PREPROCESSING
# =============================================

def pick_p_arrival(trace):
    """Deteksi P-wave arrival menggunakan STA/LTA."""
    try:
        sr = trace.stats.sampling_rate
        sta_n = int(STA_WIN * sr)
        lta_n = int(LTA_WIN * sr)
        
        if len(trace.data) < lta_n + sta_n:
            return trace.stats.starttime + 5.0
        
        cft = recursive_sta_lta(trace.data, sta_n, lta_n)
        trigger_indices = np.where(cft > TRIGGER_THRESHOLD)[0]
        
        if len(trigger_indices) > 0:
            pick_idx = trigger_indices[0]
            if pick_idx > int(2 * sr):
                return trace.stats.starttime + pick_idx / sr
        
        max_idx = np.argmax(np.abs(trace.data))
        if max_idx > 0:
            return trace.stats.starttime + max_idx / sr
    except:
        pass
    return trace.stats.starttime + 5.0

def extract_mcuquake_windows(trace, p_time):
    """Ekstraksi 7 detik sinyal dan noise, preprocessing."""
    try:
        # Signal: 7 detik setelah P
        tr_signal = trace.copy().trim(p_time, p_time + SIG_DURATION)
        # Noise: 7 detik sebelum P
        tr_noise = trace.copy().trim(p_time - NOISE_DURATION, p_time)
        
        # Detrend
        tr_signal.detrend('simple')
        tr_noise.detrend('simple')
        
        # Resample ke 100 Hz
        if tr_signal.stats.sampling_rate != SAMPLE_RATE:
            tr_signal.resample(SAMPLE_RATE)
        if tr_noise.stats.sampling_rate != SAMPLE_RATE:
            tr_noise.resample(SAMPLE_RATE)
        
        # Normalisasi dengan max absolut 9 detik setelah P
        tr_norm = trace.copy().trim(p_time, p_time + NORM_WINDOW)
        if len(tr_norm.data) > 0:
            max_val = np.max(np.abs(tr_norm.data))
        else:
            max_val = np.max(np.abs(tr_signal.data))
        if max_val == 0:
            max_val = 1.0
        
        signal_data = tr_signal.data / max_val
        noise_data = tr_noise.data / max_val
        
        # Potong/padding ke 700 sampel
        target_len = int(SAMPLE_RATE * SIG_DURATION)
        def fix_length(data):
            if len(data) > target_len:
                return data[:target_len]
            elif len(data) < target_len:
                return np.pad(data, (0, target_len - len(data)), 'constant')
            return data
        
        return fix_length(signal_data).tolist(), fix_length(noise_data).tolist()
    except Exception as e:
        return None, None

def process_file(file_path):
    """Proses satu file .mseed."""
    try:
        st = read(str(file_path))
        if len(st) == 0:
            return None
        
        # Ambil komponen Z
        trace_z = None
        for tr in st:
            if tr.stats.channel.endswith('Z'):
                trace_z = tr
                break
        if trace_z is None:
            return None
        
        # Deteksi P-wave
        p_time = pick_p_arrival(trace_z)
        
        # Ekstraksi
        signal, noise = extract_mcuquake_windows(trace_z, p_time)
        if signal is None or noise is None:
            return None
        
        # Ambil info dari nama file
        parts = file_path.stem.split('_')
        if len(parts) >= 3:
            network = parts[0]
            station = parts[1]
            event_id = '_'.join(parts[2:])
        else:
            event_id = file_path.stem
            network = 'UNK'
            station = 'UNK'
        
        return {
            'event_id': event_id,
            'network': network,
            'station': station,
            'Z': signal,
            'Z_noise': noise,
            'p_arrival': str(p_time),
            'file': file_path.name
        }
    except Exception as e:
        return None

def main():
    logger.info("="*60)
    logger.info("🚀 EKSTRAKSI WAVEFORM 20.232 FILE KE JSON")
    logger.info("="*60)
    
    # 1. Baca daftar file
    df = pd.read_csv(FILE_LIST_CSV)
    file_paths = df['path'].tolist()
    logger.info(f"📁 Total file dalam daftar: {len(file_paths)}")
    
    # Filter file yang benar-benar ada
    valid_files = []
    for p in file_paths:
        if os.path.exists(p):
            valid_files.append(p)
    logger.info(f"✅ File valid (ada di disk): {len(valid_files)}")
    
    if len(valid_files) == 0:
        logger.error("❌ Tidak ada file valid!")
        return
    
    # 2. Load JSON existing (resume)
    existing_data = {}
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r') as f:
            existing_data = json.load(f)
        logger.info(f"📂 Load JSON existing: {len(existing_data)} entries")
    
    # 3. Filter file yang belum diproses
    files_to_process = []
    for p in valid_files:
        parts = Path(p).stem.split('_')
        station = parts[1] if len(parts) >= 3 else 'UNK'
        event_id = '_'.join(parts[2:]) if len(parts) >= 3 else Path(p).stem
        key = f"{event_id}_{station}"
        if key not in existing_data:
            files_to_process.append(p)
    
    logger.info(f"📦 File baru yang akan diproses: {len(files_to_process)}")
    
    if len(files_to_process) == 0:
        logger.info("✅ Semua file sudah diproses!")
        return
    
    # 4. Proses paralel
    success = 0
    failed = 0
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_file, Path(p)): p for p in files_to_process}
        with tqdm(total=len(futures), desc="Ekstraksi", unit="file") as pbar:
            for future in as_completed(futures):
                file_path = futures[future]
                result = future.result()
                if result:
                    key = f"{result['event_id']}_{result['station']}"
                    existing_data[key] = {
                        'type': 'se',
                        'Z': result['Z'],
                        'Z_noise': result['Z_noise'],
                        'metadata': {
                            'network': result['network'],
                            'station': result['station'],
                            'p_arrival': result['p_arrival'],
                            'file': result['file']
                        }
                    }
                    success += 1
                else:
                    failed += 1
                pbar.update(1)
                
                # Simpan setiap 100 file
                if (success + failed) % 100 == 0:
                    with open(OUTPUT_JSON, 'w') as f:
                        json.dump(existing_data, f, indent=2)
    
    # 5. Simpan final
    with open(OUTPUT_JSON, 'w') as f:
        json.dump(existing_data, f, indent=2)
    
    logger.info("="*60)
    logger.info(f"✨ SELESAI! Berhasil: {success}, Gagal: {failed}")
    logger.info(f"📁 Total data di JSON: {len(existing_data)}")
    logger.info(f"📂 Output: {OUTPUT_JSON}")
    logger.info("="*60)

if __name__ == "__main__":
    main()

2026-07-03 06:41:39,932 - INFO - ============================================================
2026-07-03 06:41:39,933 - INFO - 🚀 EKSTRAKSI WAVEFORM 20.232 FILE KE JSON
2026-07-03 06:41:39,933 - INFO - ============================================================
2026-07-03 06:41:39,968 - INFO - 📁 Total file dalam daftar: 20232
2026-07-03 06:41:40,346 - INFO - ✅ File valid (ada di disk): 20232
2026-07-03 06:41:40,387 - INFO - 📦 File baru yang akan diproses: 20232


Ekstraksi:   0%|          | 0/20232 [00:00<?, ?file/s]/opt/homebrew/Caskroom/miniforge/base/envs/tf-metal/lib/python3.11/site-packages/obspy/signal/detrend.py:31: RuntimeWarning: invalid value encountered in divide
  data -= x1 + np.arange(ndat) * (x2 - x1) / float(ndat - 1)
Ekstraksi: 100%|██████████| 20232/20232 [29:58<00:00, 11.25file/s]


2026-07-03 07:11:56,396 - INFO - ============================================================
2026-07-03 07:11:56,399 - INFO - ✨ SELESAI! Berhasil: 20164, Gagal: 68
2026-07-03 07:11:56,399 - INFO - 📁 Total data di JSON: 20164
2026-07-03 07:11:56,400 - INFO - 📂 Output: /Volumes/Extreme SSD/unduhan_waveform_merged/extracted_data.json
2026-07-03 07:11:56,400 - INFO - ============================================================


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EKSTRAKSI WAVEFORM 3 KOMPONEN (Z, N, E) KE JSON
- Parallel processing
- Resume otomatis
- STA/LTA P-wave picker dari komponen Z
- Preprocessing sesuai protokol Zhi Geng
- Output: JSON dengan Z, N, E, Z_noise, N_noise, E_noise
"""

import os
import sys
import json
import numpy as np
from pathlib import Path
from obspy import read, UTCDateTime
from obspy.signal.trigger import recursive_sta_lta
from tqdm import tqdm
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import pandas as pd

# =============================================
# 1. KONFIGURASI
# =============================================

FILE_LIST_CSV = "/Volumes/Extreme SSD/unduhan_waveform_merged/file_list_merged_final.csv"
OUTPUT_JSON = "/Volumes/Extreme SSD/unduhan_waveform_merged/extracted_data_3comp.json"
LOG_FILE = "extract_3comp.log"

# Parameter MCU-Quake
SAMPLE_RATE = 100.0
SIG_DURATION = 7.0
NOISE_DURATION = 7.0
NORM_WINDOW = 9.0

# Parameter STA/LTA
STA_WIN = 1.0
LTA_WIN = 10.0
TRIGGER_THRESHOLD = 3.5

# Parallel
MAX_WORKERS = 4

# =============================================
# 2. SETUP LOGGING
# =============================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG_FILE), logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# =============================================
# 3. FUNGSI PREPROCESSING
# =============================================

def pick_p_arrival(trace):
    """Deteksi P-wave arrival menggunakan STA/LTA pada komponen Z."""
    try:
        sr = trace.stats.sampling_rate
        sta_n = int(STA_WIN * sr)
        lta_n = int(LTA_WIN * sr)
        
        if len(trace.data) < lta_n + sta_n:
            return trace.stats.starttime + 5.0
        
        cft = recursive_sta_lta(trace.data, sta_n, lta_n)
        trigger_indices = np.where(cft > TRIGGER_THRESHOLD)[0]
        
        if len(trigger_indices) > 0:
            pick_idx = trigger_indices[0]
            if pick_idx > int(2 * sr):
                return trace.stats.starttime + pick_idx / sr
        
        max_idx = np.argmax(np.abs(trace.data))
        if max_idx > 0:
            return trace.stats.starttime + max_idx / sr
    except:
        pass
    return trace.stats.starttime + 5.0

def extract_component_windows(trace, p_time):
    """
    Ekstraksi 7 detik sinyal dan noise untuk SATU komponen (Z, N, atau E).
    Return (signal_list, noise_list) atau (None, None) jika gagal.
    """
    try:
        # Signal: 7 detik setelah P
        tr_signal = trace.copy().trim(p_time, p_time + SIG_DURATION)
        # Noise: 7 detik sebelum P
        tr_noise = trace.copy().trim(p_time - NOISE_DURATION, p_time)
        
        # Detrend
        tr_signal.detrend('simple')
        tr_noise.detrend('simple')
        
        # Resample ke 100 Hz
        if tr_signal.stats.sampling_rate != SAMPLE_RATE:
            tr_signal.resample(SAMPLE_RATE)
        if tr_noise.stats.sampling_rate != SAMPLE_RATE:
            tr_noise.resample(SAMPLE_RATE)
        
        # Normalisasi dengan max absolut 9 detik setelah P
        tr_norm = trace.copy().trim(p_time, p_time + NORM_WINDOW)
        if len(tr_norm.data) > 0:
            max_val = np.max(np.abs(tr_norm.data))
        else:
            max_val = np.max(np.abs(tr_signal.data))
        if max_val == 0:
            max_val = 1.0
        
        signal_data = tr_signal.data / max_val
        noise_data = tr_noise.data / max_val
        
        # Potong/padding ke 700 sampel
        target_len = int(SAMPLE_RATE * SIG_DURATION)
        def fix_length(data):
            if len(data) > target_len:
                return data[:target_len]
            elif len(data) < target_len:
                return np.pad(data, (0, target_len - len(data)), 'constant')
            return data
        
        return fix_length(signal_data).tolist(), fix_length(noise_data).tolist()
    except Exception as e:
        return None, None

def process_file(file_path):
    """
    Proses satu file .mseed, ekstrak Z, N, E dan noise-nya.
    Return dict atau None.
    """
    try:
        st = read(str(file_path))
        if len(st) == 0:
            return None
        
        # Ambil komponen Z (wajib untuk P-pick)
        trace_z = None
        trace_n = None
        trace_e = None
        
        for tr in st:
            ch = tr.stats.channel
            if ch.endswith('Z'):
                trace_z = tr
            elif ch.endswith('N'):
                trace_n = tr
            elif ch.endswith('E'):
                trace_e = tr
        
        # Jika tidak ada Z, skip
        if trace_z is None:
            return None
        
        # Deteksi P-wave dari Z
        p_time = pick_p_arrival(trace_z)
        
        # Ekstrak untuk Z
        Z, Z_noise = extract_component_windows(trace_z, p_time)
        if Z is None or Z_noise is None:
            return None
        
        # Ekstrak untuk N (jika ada)
        N, N_noise = None, None
        if trace_n is not None:
            N, N_noise = extract_component_windows(trace_n, p_time)
        
        # Ekstrak untuk E (jika ada)
        E, E_noise = None, None
        if trace_e is not None:
            E, E_noise = extract_component_windows(trace_e, p_time)
        
        # Ambil info dari nama file
        parts = file_path.stem.split('_')
        if len(parts) >= 3:
            network = parts[0]
            station = parts[1]
            event_id = '_'.join(parts[2:])
        else:
            event_id = file_path.stem
            network = 'UNK'
            station = 'UNK'
        
        return {
            'event_id': event_id,
            'network': network,
            'station': station,
            'Z': Z,
            'Z_noise': Z_noise,
            'N': N,
            'N_noise': N_noise,
            'E': E,
            'E_noise': E_noise,
            'p_arrival': str(p_time),
            'file': file_path.name
        }
    except Exception as e:
        logger.debug(f"Error processing {file_path.name}: {e}")
        return None

def main():
    logger.info("="*60)
    logger.info("🚀 EKSTRAKSI 3 KOMPONEN (Z, N, E)")
    logger.info("="*60)
    
    # 1. Baca daftar file
    df = pd.read_csv(FILE_LIST_CSV)
    file_paths = df['path'].tolist()
    logger.info(f"📁 Total file dalam daftar: {len(file_paths)}")
    
    valid_files = []
    for p in file_paths:
        if os.path.exists(p):
            valid_files.append(p)
    logger.info(f"✅ File valid (ada di disk): {len(valid_files)}")
    
    if len(valid_files) == 0:
        logger.error("❌ Tidak ada file valid!")
        return
    
    # 2. Load JSON existing (resume)
    existing_data = {}
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r') as f:
            existing_data = json.load(f)
        logger.info(f"📂 Load JSON existing: {len(existing_data)} entries")
    
    # 3. Filter file yang belum diproses
    files_to_process = []
    for p in valid_files:
        parts = Path(p).stem.split('_')
        station = parts[1] if len(parts) >= 3 else 'UNK'
        event_id = '_'.join(parts[2:]) if len(parts) >= 3 else Path(p).stem
        key = f"{event_id}_{station}"
        if key not in existing_data:
            files_to_process.append(p)
    
    logger.info(f"📦 File baru yang akan diproses: {len(files_to_process)}")
    
    if len(files_to_process) == 0:
        logger.info("✅ Semua file sudah diproses!")
        return
    
    # 4. Proses paralel
    success = 0
    failed = 0
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_file, Path(p)): p for p in files_to_process}
        with tqdm(total=len(futures), desc="Ekstraksi 3C", unit="file") as pbar:
            for future in as_completed(futures):
                file_path = futures[future]
                result = future.result()
                if result:
                    key = f"{result['event_id']}_{result['station']}"
                    existing_data[key] = {
                        'type': 'se',
                        'Z': result['Z'],
                        'N': result['N'],
                        'E': result['E'],
                        'Z_noise': result['Z_noise'],
                        'N_noise': result['N_noise'],
                        'E_noise': result['E_noise'],
                        'metadata': {
                            'network': result['network'],
                            'station': result['station'],
                            'p_arrival': result['p_arrival'],
                            'file': result['file']
                        }
                    }
                    success += 1
                else:
                    failed += 1
                pbar.update(1)
                
                if (success + failed) % 100 == 0:
                    with open(OUTPUT_JSON, 'w') as f:
                        json.dump(existing_data, f, indent=2)
    
    # 5. Simpan final
    with open(OUTPUT_JSON, 'w') as f:
        json.dump(existing_data, f, indent=2)
    
    logger.info("="*60)
    logger.info(f"✨ SELESAI! Berhasil: {success}, Gagal: {failed}")
    logger.info(f"📁 Total data di JSON: {len(existing_data)}")
    logger.info(f"📂 Output: {OUTPUT_JSON}")
    logger.info("="*60)

if __name__ == "__main__":
    main()

2026-07-03 08:15:26,820 - INFO - ============================================================
2026-07-03 08:15:26,821 - INFO - 🚀 EKSTRAKSI 3 KOMPONEN (Z, N, E)
2026-07-03 08:15:26,821 - INFO - ============================================================
2026-07-03 08:15:27,333 - INFO - 📁 Total file dalam daftar: 20232
2026-07-03 08:15:30,044 - INFO - ✅ File valid (ada di disk): 20232
2026-07-03 08:15:30,088 - INFO - 📦 File baru yang akan diproses: 20232


Ekstraksi 3C:   0%|          | 1/20232 [00:00<2:19:31,  2.42file/s]/opt/homebrew/Caskroom/miniforge/base/envs/tf-metal/lib/python3.11/site-packages/obspy/signal/detrend.py:31: RuntimeWarning: invalid value encountered in divide
  data -= x1 + np.arange(ndat) * (x2 - x1) / float(ndat - 1)
Ekstraksi 3C: 100%|██████████| 20232/20232 [2:08:49<00:00,  2.62file/s]  


2026-07-03 10:25:11,710 - INFO - ============================================================
2026-07-03 10:25:11,712 - INFO - ✨ SELESAI! Berhasil: 20150, Gagal: 82
2026-07-03 10:25:11,712 - INFO - 📁 Total data di JSON: 20150
2026-07-03 10:25:11,712 - INFO - 📂 Output: /Volumes/Extreme SSD/unduhan_waveform_merged/extracted_data_3comp.json
2026-07-03 10:25:11,712 - INFO - ============================================================
